# Feed, individual growth and historical weather exposure

Phase 2B-2 · Highwood Feeders simulated case study

Use deliveries to track feed demand, individual allocations for sold efficiency, actual scale intervals for growth, and the archived daily grid for weather. The [management findings](../phase2b/02_FEED_GROWTH_FINDINGS.md) explain materiality, interpretation and dashboard choices.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.ticker import PercentFormatter

from feedlot_analytics.closeout_economics import sold_duration_association
from feedlot_analytics.feed_growth import (
    feed_rollup, growth_events, growth_variability, load_analysis, lot_composition_controls,
    lot_efficiency_review, matched_growth, recorded_occupancy, sold_efficiency, stage_coverage, stage_summary,
)
from feedlot_analytics.metrics import aggregate_closeout, aggregate_pen_feed
from feedlot_analytics.relations import add_columns
from feedlot_analytics.weather_associations import (
    COST, DM, TEMPERATURE, daily_feed_associations, exposure_summary, feed_calendar, feed_mix_by_date,
    growth_weather_inputs, interval_exposure, lot_associations, paired_pen_weather, sold_weather_inputs,
    weather_calendar, weather_rollup, weather_spells, within_lot_performance,
)

root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "PHASE2A_CHECKPOINT.md").is_file())
verified, delivery = load_analysis(root)
tables = verified.tables
animals = tables["animal_closeout"]
health = tables["animal_health_summary"]
lots = tables["lot_closeout"]
pen_days = tables["pen_day_feed"]
weather = tables["weather_daily"]
summary = verified.reconciliation["summary"]
pd.options.display.float_format = "{:,.3f}".format

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 120,
    "axes.labelcolor": "#243746", "axes.titleweight": "bold", "grid.alpha": .25,
})
teal, ink, ochre, slate = "#087e83", "#243746", "#a65a17", "#657d8c"

display(pd.Series({key: summary[key] for key in ("checks_passed", "checks_failed")}), pd.Series(summary["table_rows"]))
display(pd.Series({path: profile["sha256"] for path, profile in verified.source_manifest["files"].items()}))
weather_source = verified.source_manifest["weather"]
display(pd.Series({key: weather_source[key] for key in (
    "first_date", "last_date", "date_count", "raw_sha256", "clean_sha256", "grid_latitude", "grid_longitude"
)}))

## What drives feed demand and cost?

Use the native lot–pen–delivery date for ration phase. The bridge checks delivered DM and CAD against the estimated individual allocations after independent lot aggregation. Sold FCR and feed cost of gain use the Phase 2A positive-gain denominators. Individual ratios retain source precision.

In [ ]:
bridge = tables["lot_feed_reconciliation"]
overall = aggregate_closeout(animals, health).iloc[0]
pen_feed = aggregate_pen_feed(pen_days).iloc[0]
display(pd.Series(summary["feed_bridge"]), pen_feed[[DM, COST]])
display(overall[[
    "head_fcr_eligible", "fcr_gain_kg", "sold_fcr", "mean_individual_sold_fcr",
    "sold_feed_cog", "mean_individual_sold_feed_cog", "sold_adg_kg_per_day"
]])
phase = feed_rollup(delivery, by=("ration_phase",))
components = [f"{part}_{share}_share" for share in ("dm", "cost")
              for part in ("forage", "concentrate", "supplement")]
display(phase[["recorded_head_days", "delivered_dm_kg", "delivered_feed_cost_cad", DM, COST,
               "cad_per_kg_dm", *components]])
display(pen_feed[["forage_dmi_kg", "concentrate_dmi_kg", "supplement_dmi_kg",
                  "forage_cost_cad", "concentrate_cost_cad", "supplement_cost_cad"]])

In [ ]:
daily_weather = weather_calendar(weather)
daily_feed, feed_by_type = feed_calendar(pen_days, daily_weather)
monthly = feed_rollup(daily_feed, by=("month",)).join(
    daily_feed.groupby("month").feed_date.size().rename("operational_days")
)
display(feed_rollup(daily_feed.loc[daily_feed.full_month], by=("season",))[
    ["recorded_head_days", "delivered_dm_kg", "delivered_feed_cost_cad", DM, COST, "cad_per_kg_dm",
     "concentrate_dm_share"]
])
display(monthly[["operational_days", "recorded_head_days", DM, COST, "cad_per_kg_dm"]])

x = monthly.index.to_timestamp()
fig, axes = plt.subplots(2, 1, figsize=(9, 4.8), sharex=True, layout="constrained")
for axis, metric, label, color in zip(
    axes, (DM, COST), ("kg DM / head-day", "CAD / head-day"), (teal, ochre), strict=True
):
    axis.plot(x[:-1], monthly[metric].iloc[:-1], color=color, marker="o", markersize=3)
    axis.scatter(x[-1], monthly[metric].iloc[-1], color=color, marker="D", s=28)
    axis.set_ylabel(label)
    axis.grid(axis="y")
axes[0].set_title("Delivered feed intensity and cost by operational month", loc="left")
axes[0].annotate("Dec 2025 · 4 days", (x[-1], monthly[DM].iloc[-1]), xytext=(-108, -2),
                 textcoords="offset points", color=slate)
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.show()

In [ ]:
sold = sold_efficiency(animals, health)
by_breed = aggregate_closeout(animals, health, by=("breed",))
display(by_breed[[
    "breed", "head_fcr_eligible", "fcr_gain_kg", "sold_fcr", "sold_feed_cog", "sold_adg_kg_per_day"
]])
display(sold[["fcr", "lifetime_adg_kg", "financial_dof", "feed_cost_of_gain_per_kg"]]
        .describe(percentiles=[.10, .25, .50, .75, .90]))
display(sold.loc[sold.fcr_percentile.ge(.90)]
        .groupby("breed").agg(head=("animal_id", "size"), lowest_flagged_fcr=("fcr", "min")))
lot_review = lot_efficiency_review(lots, delivery)
lot_fields = ["head_fcr_eligible", "sold_fcr", "peer_median_fcr", "fcr_above_peer_median",
              "sold_feed_cog", "sold_adg_kg_per_day", DM, COST, "all_outcome_margin_per_head_placed"]
display(lot_review[lot_fields].describe())
display(lot_review.nlargest(8, "fcr_above_peer_median").set_index("lot_id")[
    ["placement_quarter", "peer_lots", *lot_fields]
])
display(within_lot_performance(sold), sold_duration_association(animals))

## At which stages does performance change?

Pair consecutive interval ADGs on the same animal. The interval uses elapsed days and excludes the previous weigh date. The late comparison has the variability of individually observed closeout intervals and includes only sales. A scheduled stage remains at risk through its date under the inclusive exit convention. Pen resampling keeps all paired animals and lots of the sampled pens together. The intervals describe sensitivity in this simulated snapshot.

In [ ]:
events = growth_events(tables["animal_scale_stage"], animals)
coverage = stage_coverage(events, animals)
stages = stage_summary(events)
wide, pairs = matched_growth(events)
variation, low_ranks = growth_variability(wide)
display(coverage, stages, pairs, variation, low_ranks)

# Show example animals from the same lot near its 10th, 50th and 90th percentiles of sold ADG.
sample_lot = lot_review.fcr_above_peer_median.idxmax()
sample_sold = sold.loc[sold.lot_id.eq(lot_review.loc[sample_lot, "lot_id"])].sort_values("lifetime_adg_kg")
examples = sample_sold.iloc[[round((len(sample_sold) - 1) * q) for q in (.10, .50, .90)]]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), layout="constrained")
for color, animal in zip((ochre, slate, teal), examples.animal_id, strict=True):
    history = events.loc[events.animal_id.eq(animal)].sort_values("weigh_date")
    axes[0].plot(history.elapsed_days_since_entry, history.weight_kg, "o-", color=color, label=animal)
axes[0].set(title=f"Example trajectories · {examples.lot_id.iloc[0]}", xlabel="Elapsed days from entry",
            ylabel="Live weight (kg)")
axes[0].legend(frameon=False, fontsize=8)
position = range(len(pairs))
axes[1].errorbar(position, pairs.paired_change, yerr=[
    pairs.paired_change - pairs.pen_bootstrap_low, pairs.pen_bootstrap_high - pairs.paired_change,
], fmt="o", capsize=4, color=teal)
axes[1].set_xticks(position, [f"{a} → {b}\nn={n:,}" for a, b, n in
                             zip(pairs.from_stage, pairs.to_stage, pairs.matched_head, strict=True)], fontsize=8)
axes[1].axhline(0, color=slate, lw=.8)
axes[1].set(title="Matched change in interval ADG", ylabel="Change (kg/day)")
plt.show()

In [ ]:
growth_fields = ["recorded", "lots", "mean_interval_adg", "sd_interval_adg", "mean_elapsed_cumulative_adg"]
for attribute in ("breed", "sex", "frame_score", "entry_weight_group"):
    display(stage_summary(events, by=("weigh_stage", attribute))[growth_fields])
display(stage_summary(events, by=("lot_id", "weigh_stage")).groupby("weigh_stage", observed=True)
        .mean_interval_adg.describe())

## Where was recorded capacity most utilized?

Recorded occupancy uses head-days divided by capacity-days on delivery-observed pen-days. Nominal capacity is counted once per pen for display. The inclusive animal ledger reconciles to the feed records, including exit days. A pen-day without a delivery row has unknown census here. Site zone and pen type describe recorded classifications.

In [ ]:
pen_month = pen_days.assign(month=pen_days.feed_date.dt.to_period("M"))
monthly_occupancy = recorded_occupancy(pen_month, by=("month",))
pen_performance = aggregate_closeout(animals, health, by=("pen_type",)).set_index("pen_type")
occupancy_fields = ["pens", "dates", "observed_pen_days", "recorded_head_days", "nominal_capacity_observed_pens",
                    "recorded_occupancy", "peak_fraction", "near_capacity_days", "above_capacity_days",
                    "overlapping_lot_days"]
for group in ("pen_type", "site_zone", "pen_id"):
    occup = recorded_occupancy(pen_days, by=(group,)).join(feed_rollup(pen_days, by=(group,))[[DM, COST]])
    with pd.option_context("display.max_rows", 45):
        display(occup[occupancy_fields + [DM, COST]])
display(monthly_occupancy[["pens", "observed_pen_days", "recorded_head_days", "recorded_occupancy"]])
display(pen_performance[["head_placed", "head_sold", "sold_adg_kg_per_day", "sold_fcr", "sold_feed_cog"]])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), layout="constrained")
for category, group in occup.join(pen_days[["pen_id", "pen_type"]].drop_duplicates().set_index("pen_id")).groupby("pen_type"):
    axes[0].scatter(group.nominal_capacity_observed_pens, group.recorded_occupancy, label=category, s=21, alpha=.8)
axes[0].set(title="Recorded utilization across 40 pens", xlabel="Nominal head capacity", ylabel="Observed capacity-days used")
axes[0].yaxis.set_major_formatter(PercentFormatter(1))
axes[0].legend(frameon=False, fontsize=7)
xm = monthly_occupancy.index.to_timestamp()
axes[1].plot(xm[:-1], monthly_occupancy.recorded_head_days.iloc[:-1], color=teal)
axes[1].scatter(xm[-1], monthly_occupancy.recorded_head_days.iloc[-1], color=teal, marker="D", s=23)
axes[1].set(title="Recorded head-days by month", ylabel="Head-days")
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[1].annotate("4 days", (xm[-1], monthly_occupancy.recorded_head_days.iloc[-1]),
                 xytext=(-42, 9), textcoords="offset points", color=slate)
plt.show()

## When did the grid record weather stress?

Cold is daily mean air temperature below 0°C. Extreme cold is daily minimum at or below −20°C. These exploratory thresholds describe reference conditions. Seasons pool complete months. The archive spans 704 dates and December 2025 covers only days 1–4. ERA5 attribution follows the documented request and acquisition provenance. The archived JSON does not identify the model. All pens share the same representative grid; shelter classifications describe the recorded lot assignment. [Provenance and day boundary](../phase2a/WEATHER_REFERENCE.md)

In [ ]:
weather_month = weather_rollup(daily_weather, by=("month", "full_month"))
display(weather_month)
display(weather_rollup(daily_weather.loc[daily_weather.full_month], by=("season",)))
for flag in ("cold", "extreme_cold"):
    spells = weather_spells(daily_weather, flag).sort_values("days", ascending=False)
    display(pd.Series({"measure": flag, "spells": len(spells), "longest_days": spells.days.max()}), spells.head(5))
display(daily_weather.loc[[daily_weather.temperature_2m_min.idxmin(), daily_weather.wind_gusts_10m_max.idxmax()],
                          ["date", TEMPERATURE, "temperature_2m_min", "wind_gusts_10m_max"]])

fig, (temperature_axis, snow_axis) = plt.subplots(2, 1, figsize=(9, 4.6), sharex=True, layout="constrained")
temperature_axis.plot(daily_weather.date, daily_weather[TEMPERATURE], color=teal, lw=.9, label="Mean air")
temperature_axis.plot(daily_weather.date, daily_weather.apparent_temperature_mean, color=slate, lw=.7,
                      alpha=.75, label="Mean apparent")
temperature_axis.axhline(0, color=ink, lw=.7, alpha=.6)
temperature_axis.set(title="Daily air and apparent temperature at the reference grid", ylabel="°C")
temperature_axis.legend(frameon=False, loc="lower right")
snow_axis.bar(daily_weather.date, daily_weather.snowfall_sum, width=1., color=slate)
snow_axis.set_ylabel("Snowfall (cm/day)")
snow_axis.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
snow_axis.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.show()

In [ ]:
animal_exposure = add_columns(
    tables["animal_weather_exposure"], animals[["animal_id", "placement_quarter", "exit_type"]],
    key=("animal_id",), cardinality="one_to_one"
)
lot_exposure = add_columns(tables["lot_weather_exposure"], lots[["lot_id", "arrival_date"]],
                           key=("lot_id",), cardinality="one_to_one")
lot_exposure["placement_quarter"] = lot_exposure.arrival_date.dt.to_period("Q")
display(exposure_summary(animal_exposure, by=("placement_quarter",)))
display(exposure_summary(animal_exposure, by=("pen_type",)))
display(exposure_summary(lot_exposure, by=("placement_quarter",)))
display(exposure_summary(lot_exposure, by=("pen_type",)))
with pd.option_context("display.max_rows", 95):
    display(lot_exposure.set_index("lot_id")[[
        "pen_type", "first_feed_date", "last_feed_date", "weather_date_count", "mean_temperature_c",
        "cold_days", "cold_day_proportion", "maximum_wind_kmh", "snowfall_exposure_cm"
    ]].sort_values("cold_day_proportion", ascending=False))

## How much of the feed association survives calendar and mix comparisons?

Match weather by feed date. First compare ratio-of-totals by cold status. The models give each of 700 operational dates in complete months equal weight. Year–month absorbs seasonal levels and slow changes. Mix terms use recorded head-day weights for ration phases, current lot DOF, pen classifications, and the placement composition of active lots. HAC intervals allow 14 or 28 days of residual dependence. The pen contrast pairs the barn and standard-open categories on the same date, then checks month deletion. Ration mix and cattle profiles can still differ within each pair.

In [ ]:
cold_feed = feed_rollup(daily_feed.loc[daily_feed.full_month], by=("cold",))
cold_type = feed_rollup(feed_by_type.loc[feed_by_type.full_month], by=("pen_type", "cold"))
display(cold_feed[["recorded_head_days", DM, COST, "cad_per_kg_dm"]])
display(cold_type[["recorded_head_days", DM, COST, "cad_per_kg_dm"]])
mix = feed_mix_by_date(delivery, animals)
daily_models = daily_feed_associations(daily_feed, mix)
display(daily_models.drop(columns="residual_df"))
display(paired_pen_weather(feed_by_type))

## Does exposure line up with growth or allocated FCR?

For scale growth, attach weather at the weighing date and over the elapsed interval (previous weighing excluded, current weighing included). The association uses the interval mean and one outcome per lot × stage. For lifetime sold ADG and FCR, use the inclusive entry–exit window of the same eligible sold animals. Compare equally weighted lots within placement month and quarter. Use individual-derived breed, sex, frame and entry-weight composition, recorded pen type, and relevant duration. The final interval includes mean elapsed interval length. Lifetime fits include inclusive financial DOF. The early-stage models avoid conditioning on future DOF.

Whole-pen deletion and deletion of every window intersecting each six-month calendar block are sensitivity ranges. Overlapping dates and shared weather limit independent evidence even at the lot grain. Interpret slopes at observed residual temperature variation and do not assign a biological response to the reanalysis.

In [ ]:
intervals = interval_exposure(events, weather, lots)
scale_lots = growth_weather_inputs(intervals, lots)
sold_lots = sold_weather_inputs(sold, tables["animal_weather_exposure"], lots)
display(pd.Series({
    "positive_elapsed_intervals": len(intervals),
    "distinct_interval_date_pairs": len(intervals[["first_interval_day", "weigh_date"]].drop_duplicates()),
    "fully_covered_scale_intervals": intervals.weather_date_count.eq(intervals.elapsed_interval_days).sum(),
    "distinct_sold_date_pairs": len(sold_lots[["first_day", "last_day"]].drop_duplicates()),
}))
traits = [col for col in lot_composition_controls(sold) if col not in ("lot_id", "mean_financial_dof")]
model_columns = ["outcome", "predictor", "increment", "calendar", "observations", "eligible_head",
                 "unique_schedules", "predictor_residual_sd", "effect", "pen_id_low", "pen_id_high",
                 "calendar_block_low", "calendar_block_high", "calendar_block_identified"]
calendars = ("placement_month", "placement_quarter")
interval_results = []
for stage, rows in scale_lots.groupby("weigh_stage", observed=True):
    controls = [*traits, "mean_interval_days"] if stage == "Closeout" else traits
    for calendar in calendars:
        fit = lot_associations(rows, predictor="mean_temperature_c", outcomes=("interval_adg",),
                               increment=1, controls=controls, calendar=calendar)
        interval_results.append(fit.assign(weigh_stage=stage))
display(pd.concat(interval_results, ignore_index=True)[["weigh_stage", *model_columns]])
lifetime_results = [
    lot_associations(sold_lots, predictor=predictor, outcomes=("sold_adg_kg_per_day", "sold_fcr"),
                     increment=increment, controls=[*traits, "mean_financial_dof"], calendar=calendar)
    for predictor, increment in (("mean_temperature_c", 1.), ("cold_day_proportion", .10))
    for calendar in calendars
]
display(pd.concat(lifetime_results, ignore_index=True)[model_columns])
display(lot_associations(sold_lots, predictor="mean_financial_dof", outcomes=("sold_fcr", "sold_adg_kg_per_day"),
                         increment=10, controls=traits)[model_columns])

Review feed phase and price together, follow repeated low ADG relative to lot peers, and show capacity only where delivery was observed. Use weather as dated context for investigating feed and growth anomalies. The [management report](../phase2b/02_FEED_GROWTH_FINDINGS.md) records the decision thresholds, case interpretation and missing evidence for validation.